# 第75章 银行客户营销转化分析

使用 UCI Bank Marketing 41,188 条真实电话营销记录，分析客户触达与定期存款转化，建立避免通话时长泄漏的营销评分基线。

## 项目背景

葡萄牙银行希望在有限呼叫容量下提高定期存款营销效率。UCI Bank Marketing 数据记录客户属性、历史联系、宏观指标和本次活动结果。duration 只有通话结束后才知道，不能用于呼叫前名单排序。

## 学习目标

- 处理分号分隔与 unknown 类别
- 分析触达次数和客户结构
- 识别 duration 的事后泄漏
- 在类别不平衡下评价模型
- 按有限呼叫容量输出 lift 分层


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| age/job/education | 客户画像 | 人口与职业类别 |
| housing/loan/default | 信贷状态 | 含 unknown |
| contact/month/day_of_week | 触达渠道与时间 | 活动字段 |
| duration | 本次通话时长 | 强泄漏，名单生成时未知 |
| campaign/pdays/previous | 联系历史 | 999 表示此前未联系 |
| poutcome | 上次活动结果 | 历史信号 |
| y | 是否认购定期存款 | 目标 |

## 数据质量检查清单

- 分隔符与字段类型
- unknown 的分布
- 目标类别不平衡
- duration 泄漏
- campaign 极端重复触达
- 时间字段不是完整时间戳


## 项目任务

1. 加载并审计
2. 分析转化与触达疲劳
3. 构建呼叫前模型
4. 评价 ROC-AUC、PR-AUC 与召回率
5. 计算 Top 10% lift
6. 输出合规营销建议


## 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 加载与营销质量审计

unknown 是原数据的显式类别，不擅自当作真实的“否”；先量化再决定处理。


In [ ]:
import numpy as np
import pandas as pd
from js import window
base_url=window.location.origin
df=pd.read_csv(f"{base_url}/datasets/bank_marketing_full.csv",sep=";")
df["target"]=(df.y=="yes").astype(int)
unknown=(df.astype(str)=="unknown").sum().sort_values(ascending=False)
print("形状:",df.shape," 转化率:",f"{df.target.mean():.2%}")
print("unknown最多字段:\n",unknown.head(8))
print("单次活动联系次数:\n",df.campaign.describe(percentiles=[.5,.9,.95,.99]).round(1))


## 2. 客群与触达诊断

这是描述性比较；活动名单本身存在选择机制，不能把组间差异解释为干预效果。


In [ ]:
job=df.groupby("job").agg(customers=("target","size"),conversion=("target","mean")).query("customers>=200").sort_values("conversion",ascending=False)
touch=pd.cut(df.campaign,[0,1,2,3,5,10,np.inf],labels=["1","2","3","4-5","6-10","11+"])
fatigue=df.groupby(touch,observed=True).agg(customers=("target","size"),conversion=("target","mean"))
print("职业客群:\n",job.round(3)); print("联系次数与转化:\n",fatigue.round(3))
print("注意：低意向客户可能被多次联系，不能据此断言多联系导致低转化。")


## 3. 构建无事后泄漏模型

排除 duration，因为生成呼叫名单时本次通话尚未发生；使用分层切分和类别权重。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
features=[c for c in df.columns if c not in ["y","target","duration"]]
X_train,X_test,y_train,y_test=train_test_split(df[features],df.target,test_size=.25,stratify=df.target,random_state=75)
cat=X_train.select_dtypes(include="object").columns.tolist(); num=[c for c in features if c not in cat]
prep=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),cat),("num",StandardScaler(),num)])
model=Pipeline([("prep",prep),("model",LogisticRegression(max_iter=700,class_weight="balanced",random_state=75))]).fit(X_train,y_train)
print("已排除事后字段 duration；训练/测试:",len(X_train),len(X_test))


## 4. 分类性能与Top-K Lift

营销名单关心有限容量内能覆盖多少转化客户，因此 PR-AUC 和 Top-K lift 比单独准确率更有用。


In [ ]:
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score
prob=model.predict_proba(X_test)[:,1]; pred=(prob>=.5).astype(int)
print(pd.Series({"ROC-AUC":roc_auc_score(y_test,prob),"PR-AUC":average_precision_score(y_test,prob),"precision@0.5":precision_score(y_test,pred),"recall@0.5":recall_score(y_test,pred)}).round(3))
ranked=pd.DataFrame({"y":y_test.to_numpy(),"p":prob}).sort_values("p",ascending=False)
for share in [.05,.10,.20]:
    top=ranked.head(int(len(ranked)*share)); lift=top.y.mean()/ranked.y.mean()
    print(f"Top {share:.0%}: 名单{len(top)}, 转化率{top.y.mean():.2%}, lift={lift:.2f}, 覆盖转化{top.y.sum()/ranked.y.sum():.1%}")


## 5. 营销策略与治理

模型排序不能替代客户同意、频控和成本收益规则。


In [ ]:
top10=ranked.head(int(len(ranked)*.1))
print(f"1. 若容量为测试客户的10%，模型名单转化率约 {top10.y.mean():.2%}，上线前需用新活动做随机对照验证增量。")
print("2. 对 campaign 设置频控并监控退订/投诉；不能从观察数据断言重复联系的因果伤害。")
print("3. 上线监控 PR-AUC、Top-K转化、覆盖率、不同客户群的触达率与投诉率。")
print("限制：数据来自历史电话活动，缺少完整成本、同意状态和时间戳；模型预测相关性，不预测营销的个体因果增量。")


## 结论与表达

- 呼叫前模型必须排除 duration。
- 不平衡营销任务应报告 PR-AUC 与容量相关 lift。
- 高响应概率不等于高增量响应。
- 频控、同意与公平触达属于部署必要条件。


## 项目验收清单

- 正确读取分号 CSV
- 明确排除 duration
- 能计算 Top 10% lift
- 能区分响应模型与 uplift/因果模型

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 UCI Bank Marketing 41,188 条真实电话营销记录，分析客户触达与定期存款转化，建立避免通话时长泄漏的营销评分基线。


### 你已经完成

- 处理分号分隔与 unknown 类别
- 分析触达次数和客户结构
- 识别 duration 的事后泄漏
- 在类别不平衡下评价模型
- 按有限呼叫容量输出 lift 分层


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 加载并审计 |
| 步骤 2 | 分析转化与触达疲劳 |
| 步骤 3 | 构建呼叫前模型 |
| 步骤 4 | 评价 ROC-AUC、PR-AUC 与召回率 |
| 步骤 5 | 计算 Top 10% lift |
| 步骤 6 | 输出合规营销建议 |


### 质量与结论提醒

- 分隔符与字段类型
- unknown 的分布
- 目标类别不平衡
- 呼叫前模型必须排除 duration。
- 不平衡营销任务应报告 PR-AUC 与容量相关 lift。
- 高响应概率不等于高增量响应。
- 频控、同意与公平触达属于部署必要条件。


### 项目交付检查

- [ ] 正确读取分号 CSV
- [ ] 明确排除 duration
- [ ] 能计算 Top 10% lift
- [ ] 能区分响应模型与 uplift/因果模型


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
